# CMR to STAC Semantics

This notebook demonstrates the **CMR → STAC** conversion semantics provided by `earthaccess`, independent of any downstream STAC tooling. You will see exactly how NASA CMR (Unified Metadata Model) fields map to [STAC 1.0.0](https://stacspec.org/) Items and Collections, and how STAC objects convert back into `earthaccess` results.

## Setup

Login so `earthaccess` can build an authenticated registry for indirect (HTTPS) access. Conversion itself is pure metadata work, but searching needs a session.

In [ ]:
import earthaccess

auth = earthaccess.login()
auth

## 1. Search granules

`search_data()` returns a lazy `GranuleResults` container. We grab a small batch of HLS granules.

In [ ]:
granules = earthaccess.search_data(
    short_name="HLSL30",
    temporal=("2025-01-01", "2025-01-20"),
    bounding_box=(40.19, 6.24, 42.18, 7.24),
    count=3,
)
granules

## 2. What does CMR give us?

A `DataGranule` is a `dict` subclass whose raw CMR payload lives under `["umm"]` (Unified Metadata Model) and `["meta"]` (CMR bookkeeping). This is the input to conversion.

In [ ]:
granule = granules[0]

print("CMR concept-id:", granule["meta"]["concept-id"])
print("GranuleUR:", granule["umm"]["GranuleUR"])
print("UMM keys:", list(granule["umm"].keys()))
print("Temporal extent:", granule["umm"].get("TemporalExtent"))

## 3. Convert a granule to a STAC Item

Two equivalent entry points:

- `granule.to_stac()` → a `pystac.Item` (or pass `access="s3"` to prefer S3 asset hrefs)
- `earthaccess.stac.umm_granule_to_stac_item(granule)` → the underlying STAC Item `dict`

Both build the same STAC 1.0.0 `Feature`.

In [ ]:
item = granule.to_stac()
print("STAC object type:", item.STAC_OBJECT_TYPE)
print("id:", item.id)
print("stac_version:", item.to_dict()["stac_version"])
print("geometry type:", item.geometry["type"])
print("bbox:", item.bbox)
print("datetime:", item.properties.get("datetime"))
print("cmr:concept_id:", item.properties.get("cmr:concept_id"))
print("assets:", {k: v.href for k, v in item.assets.items()})

### The field mapping

The same conversion as a plain `dict`, for inspection or batch processing without creating result objects:

| CMR UMM field | STAC field |
|---|---|
| `GranuleUR` | `properties.granule_ur`, `id` |
| `TemporalExtent` | `properties.datetime`, `start_datetime`, `end_datetime` |
| `SpatialExtent` | `geometry`, `bbox` |
| `RelatedUrls` | `assets` (roles: data / metadata / thumbnail) |
| `CloudCover` | `properties.eo:cloud_cover` (+ EO extension) |
| `meta.concept-id` | `properties.cmr:concept_id` |

In [ ]:
import json

from earthaccess.stac import umm_granule_to_stac_item

item_dict = umm_granule_to_stac_item(granule)
print(json.dumps(item_dict, indent=2)[:1800])

## 4. Batch conversion

`GranuleResults.to_stac()` converts every cached result to STAC objects in one call.

In [ ]:
items = granules.to_stac()
print(f"converted {len(items)} granules")
for it in items:
    print(" -", it.id, "| datetime:", it.properties.get("datetime"))

## 5. Convert a collection to a STAC Collection

Collections follow the same idea: `search_datasets()` → `DataCollection` → STAC `Collection`.

In [ ]:
collections = earthaccess.search_datasets(short_name="HLSL30", count=1)
collection = collections[0]

stac_collection = collection.to_stac()
print("STAC object type:", stac_collection.STAC_OBJECT_TYPE)
print("id:", stac_collection.id)
print("description:", stac_collection.description[:80], "...")
print("extent spatial bbox:", stac_collection.extent.spatial.bboxes)
print("cmr:concept_id:", stac_collection.extra_fields.get("cmr:concept_id"))

### Raw UMM collection → STAC Collection

`earthaccess.stac.umm_collection_to_stac_collection` does the same on the raw UMM dict.

In [ ]:
from earthaccess.stac import umm_collection_to_stac_collection

coll_dict = umm_collection_to_stac_collection(collection)
print("id:", coll_dict["id"])
print("keys:", sorted(coll_dict.keys()))

## 6. Round-trip: STAC → CMR

`earthaccess` also imports external STAC Items/Collections back into `DataGranule` / `DataCollection`, so STAC objects from any catalog can be used with `download()` / `open()`.

In [ ]:
from earthaccess.stac import stac_item_to_data_granule

item_json = json.loads(json.dumps(item.to_dict()))
roundtrip = stac_item_to_data_granule(item_json, cloud_hosted=True)

print("round-tripped GranuleUR:", roundtrip["umm"]["GranuleUR"])
print("related URLs:", [l.get("URL") for l in roundtrip["umm"].get("RelatedUrls", [])])

## Summary

- `DataGranule.to_stac()` / `DataCollection.to_stac()` give STAC objects straight from search results.
- `earthaccess.stac.*` exposes the raw `umm → stac` conversions for batch/offline use.
- `stac_item_to_data_granule()` / `stac_collection_to_data_collection()` close the loop back to `earthaccess`.

To plug STAC items into the wider ecosystem (e.g. `odc-stac`), see the
[Opening STAC-cataloged cloud data](../odc-stac-cmr.ipynb) notebook.